**IMPORTANT!** Before beginning this lab, be sure to **make your own copy** of the notebook and name it something like "Lastname - Lab 10".

# Lab 10: Unsupervised Learning — K-Means Clustering

In lecture we studied **unsupervised learning**: the idea that structure or patterns can be discovered in data *without* pre-existing labels. The most widely used unsupervised technique is **k-means clustering**, which partitions a dataset into $k$ groups (clusters) such that data points within the same cluster are as similar to each other as possible.

In this lab you will:

1. Load and explore a real NBA player statistics dataset from the 2022-23 season.
2. Cluster players based on **shooting statistics** (2P, 3P, eFG%) using k-means and interpret the results.
3. Cluster players based on **all-around statistics** (PTS, AST, TRB, STL, BLK) and determine the optimal number of clusters using the **average silhouette score**.
4. Interpret your clusters in basketball terms and identify interesting groups of players.


## Setup: Downloading Data

Run the cell below to download the data file needed for this lab from GitHub.

In [ ]:
import requests
from pathlib import Path
import polars as pl

# Force all columns to be shown when printing a DataFrame
pl.Config.set_tbl_cols(-1)

# Base URL for the lab repository
BASE_URL = 'https://raw.githubusercontent.com/JuanCab/csis446_lab10/refs/heads/main/'

# Dataset filename
local_filename = "NBA_player_stats.csv"

# Check if the file already exists to avoid re-downloading
if not Path(local_filename).is_file():
    # Data URL for the dataset
    url = BASE_URL + "data/" + local_filename
    print(f"Downloading dataset from {url}...")
    # Download the dataset using requests library
    r = requests.get(url)
    # Check if the request was successful, if not, raise an error
    r.raise_for_status()

    # Save the content to a local file
    with open(local_filename, "wb") as f:
        f.write(r.content)
        print(f"Dataset downloaded and saved as '{local_filename}'.")
else:
    print(f"Dataset '{local_filename}' already exists. Skipping download.")

## Part A: Exploring the NBA Player Statistics Dataset

### Background (Gaining Business Understanding and Data Understanding)

We have a dataset with statistics about National Basketball Association (NBA) players. The dataset is from the 2022-23 NBA season and was originally sourced from [basketball-reference.com](https://www.basketball-reference.com/). The authors pre-processed the data so that most counting statistics are expressed as **per-game averages** (PGA) so that players who missed games due to injury can still be compared fairly.

Note that a player can appear **more than once** in the dataset if they were traded mid-season — their statistics are tracked per player/team combination.


### Step A.1: Load and Explore the Dataset

In [ ]:
# Load the dataset from the local file into a Polars DataFrame
playerstats_df = pl.read_csv(local_filename)

print(f'Player Statistics DataFrame has shape: {playerstats_df.shape}')
playerstats_df.head()


Here is a reference table describing all columns in the dataset:

| Column | Description | Column | Description | Column | Description |
|--------|-------------|--------|-------------|--------|-------------|
| **Player** | Name of the player | **FG%** | Field Goal Percentage | **FT%** | Free Throw Percentage |
| **Pos** | Position | **3P** | 3-Point Field Goals (PGA)| **ORB** | Offensive Rebounds (PGA)|
| **Age** | Player's age on Feb 1 of the season | **3PA** | 3-Point Field Goal Attempts (PGA)| **DRB** | Defensive Rebounds (PGA)|
| **Tm** | Team | **3P%** | 3-Point Field Goal Percentage | **TRB** | Total Rebounds (PGA)|
| **G** | Games Played | **2P** | 2-Point Field Goals (PGA)| **AST** | Assists (PGA)|
| **GS** | Games Started | **2PA** | 2-Point Field Goal Attempts (PGA)| **STL** | Steals (PGA)|
| **MP** | Minutes Played (PGA) | **2P%** | 2-Point Field Goal Percentage | **BLK** | Blocks (PGA)|
| **FG** | Field Goals (PGA) | **eFG%** | Effective Field Goal Percentage | **TOV** | Turnovers (PGA) |
| **FGA** | Field Goal Attempts (PGA)| **FT** | Free Throws (PGA)| **PF** | Personal Fouls (PGA)|
| | | **FTA** | Free Throw Attempts (PGA)| **PTS** | Points scored (PGA) |


**Question 1**: Given the variety of statistics available, do you think there is a single statistic that best defines the "best" NBA player? Why or why not?

```
ANSWER HERE
```

### Step A.2: Understanding the Shooting Statistics and Identifying the Best Scorers

In basketball, players can score different numbers of points per game by making 2-point field goals or 3-point field goal depending on where they shoot from when sinking a ball in the basket (there are also free throws, but we are not focusing on them).

![Shooting Statistics](https://raw.githubusercontent.com/JuanCab/csis446_lab10/refs/heads/main/images/ShootingStats.png)

*An illustration of shooting statistics. This figure was taken from **Machine Learning with zyLabs** by Aimee Schwab-McCoy.*

The three shooting statistics we will focus on are:

- **2P**: Average number of 2-point field goals made per game
- **3P**: Average number of 3-point field goals made per game
- **eFG%**: Effective field goal percentage has been computed for us from **2P**, **3P**, **2PA**, and **3PA** (**2PA** and **3PA** are the average number of 2-point and 3-point field goal attempts per game, respectively). It  adjusts for the fact that a 3-point field goal is worth more and is calculated as:$$eFG\% = \frac{2P + 1.5 \times 3P}{2PA + 3PA}$$.

Let's sort the data by average per-game points scored (**PTS**) and pull out the top 200 players.

In [ ]:
# Sort the DataFrame by average per-game points scored (descending)
playerstats_df = playerstats_df.sort('PTS', descending=True)

# Select the columns to pull out for the top 200 players.
cols = ['Player', 'Pos', 'Tm', 'PTS', '3P', '2P', 'eFG%', '3P%', '2P%']

# Pull out the top 200 players by average per-game points scored.
top200_df = playerstats_df.select(cols).head(200)

# Display the top 5 players from this new DataFrame
top200_df.head()

**Question 2**: To help you gain some data understanding beyond just looking at the statistics, the cell below computes the correlation matrix and plots a heatmap for the three shooting statistics (2P, 3P, eFG%). Run it and interpret the results. Are any of the correlations surprising to you?

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute the correlation matrix for the three shooting statistics
corr_df = top200_df.select(['2P', '3P', 'eFG%']).to_pandas().corr()
print(corr_df)

# Plot a heatmap of the correlation matrix
plt.figure(figsize=(4, 4))
sns.heatmap(corr_df, annot=True, center=0)
plt.title('Correlation: Shooting Statistics')
plt.show()


```
ANSWER HERE
```

**Question 3**: Run the cell below to produce a pairplot of the three shooting statistics. Do the scatter plots support your conclusions from the correlation matrix?

In [ ]:
# Pairplot of the three shooting statistics
sns.pairplot(data=top200_df.select(['2P', '3P', 'eFG%']).to_pandas(), height=2)
plt.suptitle('Pairplot: Shooting Statistics (Top 200 Scorers)', y=1.02)
plt.show()

```
ANSWER HERE
```

**Exercise 1** *(you write the code)*: Instead of selecting the top 200 players by points per game, select the top 200 players by **3-point field goals made per game** (`3P`). Then compute the correlation matrix and produce a pairplot for the same three statistics (2P, 3P, eFG%) for this new group of players.

How do the correlation coefficients compare to the previous results?

In [ ]:
# This time we will pull the top 200 players based on 3-point shots per 
# game instead of points per game.

# TODO: Sort by 3-point shots per game
playerstats_df = ...

# Select the columns to pull out for the top 200 players.
cols = ['Player', 'Pos', 'Tm', 'PTS', '3P', '2P', 'eFG%', '3P%', '2P%']

# TODO: Pull out the top 200 players by 3P
top200_3P_df = ...

# TODO: Correlation matrix
corr_3P_df = ...
print(corr_3P_df)

# Plot the heatmap of the correlation matrix for the top 200 3P shooters
plt.figure(figsize=(4, 4))
sns.heatmap(corr_3P_df, annot=True, center=0)
plt.title('Correlation: Top 200 3P Shooters')
plt.show()

# Show the pairplot for the top 200 3P shooters
sns.pairplot(data=top200_3P_df.select(['2P', '3P', 'eFG%']).to_pandas(), height=2)
plt.suptitle('Pairplot: Top 200 3P Shooters', y=1.02)
plt.show()


```
ANSWER HERE
```

## Part B: K-Means Clustering on Shooting Statistics

### Background

Recall that K-means clustering partitions data into $k$ groups by iteratively:

1. Assigning each point to its nearest centroid (cluster center).
2. Recomputing each centroid as the mean of all points assigned to it.

Before clustering, we **standardize** the features so that no single feature dominates the distance calculations simply because it happens to be measured on a larger scale.

### Step B.1: Prepare the Data for Clustering

We'll start by reloading the data cleanly and selecting the top 200 scorers.

In [ ]:
# Pull out the top 200 players by average per-game points scored.
top200_df = playerstats_df.select(cols).head(200)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Step 1: Select the three shooting features (convert to pandas for sklearn)
X = top200_df.select(['3P', '2P', 'eFG%']).to_pandas()

# Step 2: Standardize the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Step 3: Convert back to a Pandas DataFrame with the original column
# names (since we are using sklearn, we need to have it as a pandas
# DataFrame)
X = pd.DataFrame(X, columns=['3P', '2P', 'eFG%'])

print('First 5 rows of standardized features:')
print(X.head())


```
ANSWER HERE
```

### Step B.2: Fit K-Means with k = 3

We'll start by fitting a k-means model with $k = 3$ clusters and visualizing the cluster membership distribution.

**Exercise 2** *(you write the code)*: Fit a k-means model with $k = 3$ clusters using the standardized features. T

In [ ]:
from sklearn.cluster import KMeans

# TODO: Build k-means model with k=3 clusters using a random_state of 
# 123 and n_init of 10.
k = 3
kmeans_mdl = ...
# TODO: Fit k-means to the standardized features
kmeans_results = ...

# Print the cluster centroids and inertia
print('Cluster centroids (standardized feature space):')
print(kmeans_results.cluster_centers_)
print(f'\nInertia: {kmeans_results.inertia_:.4f}')

### Step B.3: Visualize the Clusters

A **pairplot** colored by cluster membership lets us see how the clusters differ across all pairs of features simultaneously. Note that the diagonal now shows kernel density estimates (KDEs) rather than histograms when `hue` is used.

**A Note about Inertia**: Notice we printed the value of the `inertia_` attribute of the fitted model. This is the sum of squared distances of samples to their closest cluster center. It is a measure of how well the clusters fit the data (lower is better).


In [ ]:
# Add the cluster labels to top200_df as strings
# (strings make seaborn treat clusters as categorical so that
# they are colored differently in the count plot)
top200_df = top200_df.with_columns(
    pl.Series('clusters', kmeans_results.labels_.astype(str))
).sort('clusters')

# Bar chart of cluster membership counts
plt.figure(figsize=(6, 4))
sns.countplot(data=top200_df.to_pandas(), x='clusters', hue='clusters')
plt.xlabel('Cluster')
plt.ylabel('Number of Players')
plt.title('Cluster Membership: k=3 (Shooting Stats)')
plt.show()

In [ ]:
# Pairplot colored by cluster
sns.pairplot(data=top200_df.select(['3P', '2P', 'eFG%', 'clusters']).to_pandas(),
             hue='clusters', height=2)
plt.suptitle('Pairplot by Cluster: Shooting Statistics', y=1.02)
plt.show()

Since we have exactly 3 features, we can render a **3D scatter plot** using plotly to visualize the clusters in three-dimensional space. This can sometimes reveal structure that is not as clear in 2D pairplots.  I believe plotly is installed in Google Colab, but if you run into an error, you can install it using `!pip install plotly` in a code cell.

In [ ]:
# This cell was generated using Claude as I am not a plotly expert, I just
# wanted an interactive 3D scatter plot to visualize the clusters in 3D space.

import plotly.graph_objects as go

# Map cluster labels to colors and marker symbols
palette = {'0': 'steelblue', '1': 'darkorange', '2': 'seagreen'}
symbols = {'0': 'circle', '1': 'diamond', '2': 'square'}

fig = go.Figure()

for cluster_label in sorted(top200_df['clusters'].unique().to_list()):
    group = top200_df.filter(pl.col('clusters') == cluster_label)
    fig.add_trace(go.Scatter3d(
        x=group['3P'].to_list(),
        y=group['2P'].to_list(),
        z=group['eFG%'].to_list(),
        mode='markers',
        name=f'Cluster {cluster_label}',
        marker=dict(
            size=5,
            color=palette[cluster_label],
            symbol=symbols[cluster_label],
            opacity=0.6
        )
    ))

fig.update_layout(
    title='K-Means Clusters: NBA Shooting Statistics (k=3)',
    width=800,   # change these to resize the plot
    height=600,
    scene=dict(
        xaxis_title='3P (3-pt FG/game)',
        yaxis_title='2P (2-pt FG/game)',
        zaxis_title='eFG%'
    ),
    legend_title='Cluster'
)
fig.show()


**Question 5**: The cell below creates a boxplot of `eFG%` separated by cluster. Run it and describe how cluster 1 differs from the other two clusters with respect to effective field goal percentage.

In [ ]:
# Boxplot: eFG% by cluster
sns.boxplot(data=top200_df.select(['eFG%', 'clusters']).to_pandas(),
            x='eFG%', y='clusters', hue='clusters')
plt.title(
    'eFG% by Cluster')
plt.show()


```
ANSWER HERE
```

**Exercise 3** *(you write the code)*: Create a boxplot of `3P` (3-point field goals made per game) separated by cluster. Based on this plot, which cluster is characterized by a high number of 3-point field goals made per game?

In [ ]:
# TODO:Create a boxplot of 3P by cluster 


```
ANSWER HERE
```

---
## Part C: K-Means Clustering on All-Around Player Statistics

### Background

K-means can handle data with more than 3 dimensions. We'll now cluster NBA players using **five all-around statistics** that capture different aspects of player performance:

| Feature | Description |
|---------|-------------|
| **PTS** | Average points scored per game |
| **AST** | Average assists per game (passes leading directly to a score) |
| **TRB** | Average total rebounds per game (offensive + defensive) |
| **STL** | Average steals per game (taking the ball from an opponent) |
| **BLK** | Average blocks per game (deflecting an opponent's shot) |

### Step C.1: Load and Prepare the Data


In [ ]:
# Load the data and select the top 200 players by points per game
playerstats_df = pl.read_csv('data/NBA_player_stats.csv')
playerstats_df = playerstats_df.sort('PTS', descending=True)

cols = ['Player', 'Pos', 'Tm', '2P', '3P', 'eFG%', 'PTS', 'AST', 'TRB', 
        'STL', 'BLK']
top200_df = playerstats_df.select(cols).head(200)

top200_df.head()


**Question 6**: Run the cell below to compute the correlation matrix for the five all-around statistics. Identify the two strongest correlations (positive or negative) and provide your best explanation for each.

In [ ]:
# Correlation matrix for the five all-around statistics
corr_df = top200_df.select(['PTS', 'AST', 'TRB', 'STL', 'BLK']).to_pandas().corr()
print(corr_df)

plt.figure(figsize=(5, 4))
sns.heatmap(corr_df, annot=True, center=0)
plt.title('Correlation: All-Around Statistics')
plt.show()


```
ANSWER HERE
```

### Step C.2: Standardize Features and Fit K-Means

**Exercise 4** *(you write the code)*: Finish the code in the cell below to perform the following steps:

1. Extract the five all-around statistics (`PTS`, `AST`, `TRB`, `STL`, `BLK`) from `top200_df` into a variable `X`.
2. Standardize `X` using `StandardScaler` and convert it back to a DataFrame with the original column names.
3. Fit a `KMeans` model with `k=4` clusters, `random_state=123`, and `n_init=10`.
4. Produce a bar chart of cluster membership.

In [ ]:
# TODO: Extract features (convert to pandas for sklearn)
X = ...

# TODO: Standardize those features using sklearn's StandardScaler and
# convert back to a pandas when done
scaler = ...
X = scaler.fit_transform(X)
X = pd.DataFrame(...)

# TODO: Build k-means model with 4 clusters using a random_state of 123 
# and n_init of 10.
k = 4
kmeans_mdl = ...
# TODO: Fit the model to the standardized features
kmeans_results = ...

# Print the cluster centroids and inertia
print('Centroids (standardized feature space):')
print(kmeans_results.cluster_centers_)
print(f'Inertia: {kmeans_results.inertia_:.4f}')

# Add labels to a copy and plot
top200_clustered_df = top200_df.with_columns(
    pl.Series('clusters', kmeans_results.labels_.astype(str))
).sort('clusters')

# TODO: Bar chart of cluster membership counts

### Step C.3: Pairplot of All-Around Statistics by Cluster

With 5 features we can't do a 3D scatter plot, but a pairplot gives us a good overview of how the clusters differ across all pairs of statistics.

In [ ]:
# Pairplot of all-around statistics colored by cluster
cols = ['PTS', 'AST', 'TRB', 'STL', 'BLK', 'clusters']
sns.pairplot(data=top200_clustered_df.select(cols).to_pandas(), hue='clusters')
plt.suptitle('Pairplot by Cluster: All-Around Statistics (k=4)', y=1.02)
plt.show()


### Step C.4: Determine the Optimal Number of Clusters

Rather than guessing $k$, we can use the **average silhouette score** to evaluate how well-separated the clusters are for different values of $k$. Recall from lecture that the silhouette score for a single point $i$ is:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))}$$

where $a(i)$ is the mean intra-cluster distance and $b(i)$ is the mean distance to the nearest neighboring cluster. A higher average silhouette score (closer to 1) indicates better-defined clusters.

**Exercise 5** *(you write the code)*: Complete the code below to compute the average silhouette score for $k = 2, 3, \ldots, 10$ clusters. Store the results in the list `avg_silhouette`. The code will then plot the average silhouette score vs. $k$.  Use this plot to identify the optimal number of clusters.

In [ ]:
from sklearn.metrics import silhouette_score

# Store average silhouette scores for k=2..10
avg_silhouette = []
k_values = range(2, 11)

# Loop over k values, fit k-means, compute silhouette score, and store it
for k in k_values:
    # TODO: Create k-means clustering model for this k using random_state=123
    # and n_init=10, then fit X with the model.
    cluster = ...
    # TODO:Compute the silhouette score for this k
    s = ...
    # Append the silhouette score to the list
    avg_silhouette.append(s)
    # Print the silhouette score for this k
    print(f'k={k}: avg silhouette = {s:.4f}')

# Plot average silhouette score vs k
plt.figure(figsize=(7, 4))
plt.plot(k_values, avg_silhouette, marker='o')
plt.xlabel('k (number of clusters)', fontsize=14)
plt.ylabel('Average Silhouette Score', fontsize=14)
plt.title('Average Silhouette Score vs. k')
plt.show()


**Question 7**: Based on the average silhouette score plot, what is the optimal number of clusters for the all-around statistics dataset? Explain your reasoning.

```
ANSWER HERE
```

**Exercise 6** *(you write the code)*: Re-fit the k-means model using the optimal number of clusters you identified above. Update `top200_clustered_df` with the new cluster labels and reproduce the pairplot.

In [ ]:
# TODO: Re-fit KMeans with the optimal k (again, use random_state=123 
# and n_init=10 for reproducibility)
k = ...
kmeans_mdl = KMeans(n_clusters=k, random_state=123, n_init=10)
kmeans_results = kmeans_mdl.fit(X)

# Add the cluster labels to top200_df as strings
top200_clustered_df = top200_df.with_columns(
    pl.Series('clusters', kmeans_results.labels_.astype(str))
).sort('clusters')

# TODO: Plot count plot (aka histogram) of cluster membership

# TODO: Also produce a Pairplot
cols = ['PTS', 'AST', 'TRB', 'STL', 'BLK', 'clusters']



**Question 8**: Looking at the pairplot above, describe each cluster in plain English. What type of player does each cluster appear to represent? *(Think about which statistics are high or low for each cluster.)*

```
ANSWER HERE
```

### Step C.5: Examine the Players in Each Cluster

Let's print the players in each cluster along with their key statistics to see if our interpretation holds up. Note that some players appear more than once because they were traded mid-season and have separate entries for each team.

In [ ]:
# Print the players and key stats for each cluster
for cluster in sorted(top200_clustered_df['clusters'].unique().to_list()):
    print(f'\n{"="*60}')
    print(f'Cluster {cluster}:')
    print(f'{"="*60}')
    subset = top200_clustered_df.filter(pl.col('clusters') == cluster)
    print(subset.select(['Player', 'Tm', 'PTS', 'AST', 'TRB', 'STL', 'BLK']))
    print()


Below we compute and display the **mean** of each statistic (`PTS`, `AST`, `TRB`, `STL`, `BLK`) for each cluster. This gives you the cluster centroids in the original (un-standardized) feature space, which are easier to interpret.


In [ ]:
# Here we use polar's groupby and agg (aggregate)to compute the mean of 
# each statistic for each cluster, then sort by cluster label.
cluster_means = (
    top200_clustered_df
    .group_by('clusters')
    .agg([pl.col(c).mean().round(3).alias(c) for c in ['PTS', 'AST', 'TRB', 'STL', 'BLK']])
    .sort('clusters')
)
print('Mean statistics per cluster (original scale):')
print(cluster_means)


**Question 9**: Based on the cluster mean statistics, which cluster has the highest average points per game? Which has the highest average rebounds per game? Do these results align with your descriptions in Question 8?

```
ANSWER HERE
```

---

## What to Submit

Once you have completed all parts of this lab:

1. Make sure all cells are run and all output is visible (do a final **Runtime → Run all** to confirm).
2. Go to **File → Download → Download .ipynb** to save the notebook to your computer.
3. Submit the `.ipynb` file to the **Lab 9** dropbox on D2L.

You will be graded on:
- **Correct code** (all cells run without error)
- **Written answers** that demonstrate understanding of the concepts
- **Prediction cells** (Questions 6 and 8) filled in *before* running the corresponding experiments